In [1]:
from adagio.execute import execute_pipeline
from adagio.model.pipeline import AdagioPipeline
from adagio.model.arguments import AdagioArguments

In [2]:
pipeline = AdagioPipeline.model_validate_json(open('./examples/simple.json').read())

In [3]:
args = pipeline.signature.to_default_arguments()
args

inputs:
    sample_metadata: '<fill me>'
    table: '<fill me>'
parameters:
    metric: 'canberra'
    compare: '<fill me>'
outputs:
    summary: '<fill me>'
    distance_matrix1: '<fill me>'
    beta-group: '<fill me>'
    distance_matrix2: '<fill me>'
    pcoa: '<fill me>'
    emperor: '<fill me>'

In [4]:
args.inputs['sample_metadata'] = '/home/evan/Downloads/sample-metadata.tsv'
args.inputs['table'] = '/home/evan/Downloads/table.qza'
args.parameters['compare'] = 'body-site'
for key in args.outputs:
    args.outputs[key] = key
args

inputs:
    sample_metadata: '/home/evan/Downloads/sample-metadata.tsv'
    table: '/home/evan/Downloads/table.qza'
parameters:
    metric: 'canberra'
    compare: 'body-site'
outputs:
    summary: 'summary'
    distance_matrix1: 'distance_matrix1'
    beta-group: 'beta-group'
    distance_matrix2: 'distance_matrix2'
    pcoa: 'pcoa'
    emperor: 'emperor'

In [5]:
from qiime2.sdk.parallel_config import ParallelConfig
from qiime2.sdk.parallel_config import _get_vendored_config_path
!cat {_get_vendored_config_path()}
config = ParallelConfig(_get_vendored_config_path())

[parsl]
strategy = "None"

[[parsl.executors]]
class = "ThreadPoolExecutor"
label = "tpool"
max_threads = 15

[[parsl.executors]]
class = "HighThroughputExecutor"
label = "default"
max_workers = 15

[parsl.executors.provider]
class = "LocalProvider"


In [6]:
with config:
    execute_pipeline(pipeline, args)

/home/evan/.conda/envs/adagio-amplicon-2025.7/lib/python3.10/site-packages/emperor/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


SCHEDULED: load_metadata('/home/evan/Downloads/sample-metadata.tsv')
SCHEDULED: load_input('/home/evan/Downloads/table.qza')
SCHEDULED: feature_table summarize
SCHEDULED: diversity beta
SCHEDULED: diversity beta_group_significance
SCHEDULED: diversity beta
SCHEDULED: diversity pcoa
SCHEDULED: emperor plot
SCHEDULED: summary.save('summary')
SCHEDULED: distance_matrix1.save('distance_matrix1')
SCHEDULED: beta-group.save('beta-group')
SCHEDULED: distance_matrix2.save('distance_matrix2')
SCHEDULED: pcoa.save('pcoa')
SCHEDULED: emperor.save('emperor')
DONE: load_input('/home/evan/Downloads/table.qza')
SCHEDULED: diversity_lib beta_passthrough
SCHEDULED: diversity_lib beta_passthrough
DONE: diversity_lib beta_passthrough
DONE: diversity beta
DONE: load_metadata('/home/evan/Downloads/sample-metadata.tsv')
DONE: distance_matrix1.save('distance_matrix1')


/home/evan/.conda/envs/adagio-amplicon-2025.7/lib/python3.10/site-packages/q2_diversity/_beta/_visualizer.py:174: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pairs_summary = pd.concat([pairs_summary, group_pairs_summary])


DONE: diversity beta_group_significance
DONE: beta-group.save('beta-group')
DONE: diversity_lib beta_passthrough
DONE: diversity beta


/home/evan/.conda/envs/adagio-amplicon-2025.7/lib/python3.10/site-packages/skbio/stats/ordination/_principal_coordinate_analysis.py:146: RuntimeWarning: The result contains negative eigenvalues. Please compare their magnitude with the magnitude of some of the largest positive eigenvalues. If the negative ones are smaller, it's probably safe to ignore them, but if they are large in magnitude, the results won't be useful. See the Notes section for more details. The smallest eigenvalue is -9984.87758428538 and the largest is 99347.1154093905.
  warn(


DONE: diversity pcoa
DONE: emperor plot
DONE: distance_matrix2.save('distance_matrix2')
DONE: emperor.save('emperor')
DONE: pcoa.save('pcoa')
DONE: feature_table summarize
DONE: summary.save('summary')


In [7]:
!ls -lh $(echo '*.qz[a|v]')

-rw-rw-r-- 1 evan evan 363K Sep 25 17:05 beta-group.qzv
-rw-rw-r-- 1 evan evan  56K Sep 25 17:05 distance_matrix1.qza
-rw-rw-r-- 1 evan evan  56K Sep 25 17:06 distance_matrix2.qza
-rw-rw-r-- 1 evan evan 859K Sep 25 17:06 emperor.qzv
-rw-rw-r-- 1 evan evan  76K Sep 25 17:06 pcoa.qza
-rw-rw-r-- 1 evan evan 442K Sep 25 17:06 summary.qzv
